In [1]:
import email
import imaplib
from io import BytesIO

import pandas as pd

In [2]:
prev = 0


def pretty(label: str, completed: float, total: int, length: int = 30) -> None:
    global prev
    print(
        " " * (prev * 2)
        + f"\r{label} ["
        + "=" * int(completed / total * length)
        + "-" * int((total - completed) / total * length)
        + f"] {int(completed / total * 100)}%",
        end="",
    )
    prev = length


def prettyPrintReplace(label: str) -> None:
    global prev
    print(" " * (prev * 2) + f"\r{label}", end="")
    prev = len(label)

In [3]:
# Login credentials
IMAP_HOST = "imap.gmail.com"
EMAIL_USER = "ankushmisal7387@gmail.com"
EMAIL_PASS = "lnfv cnrn yrju ukom"

In [4]:
subject_to_search = "STATE CODE - 27 | "

In [5]:
# Connect and login
mail = imaplib.IMAP4_SSL(IMAP_HOST)
mail.login(EMAIL_USER, EMAIL_PASS)
mail.select("inbox")

# Search emails by subject
status, messages = mail.search(None, f'(SUBJECT "{subject_to_search}")')

In [6]:
rawGRN = pd.DataFrame()
rawPR = pd.DataFrame()
totalMailParsed = 0
total_mail_to_parse = len(messages[0].split())
if status == "OK":
    for num in messages[0].split():
        typ, data = mail.fetch(num, "(RFC822)")
        msg = email.message_from_bytes(data[0][1])

        # Extract only selected attachments
        for part in msg.walk():
            content_disposition = str(part.get("Content-Disposition") or "")
            prettyPrintReplace(
                f"{totalMailParsed}/{total_mail_to_parse}, GRN: {len(rawGRN)}, PR: {len(rawPR)}"
            )

            if part.get_content_maintype() == "multipart":
                continue

            if "attachment" in content_disposition:
                filename = part.get_filename()
                # if filename and filename.lower() in [f.lower() for f in allowed_files]:
                if filename.lower().endswith("25.csv") and filename.lower().startswith(
                    "grn"
                ):
                    file_data = part.get_payload(decode=True)
                    file_stream = BytesIO(file_data)
                    rawGRN = pd.concat([rawGRN, pd.read_csv(file_stream)])
                elif filename.lower().endswith(
                    "25.csv"
                ) and filename.lower().startswith("pr"):
                    file_data = part.get_payload(decode=True)
                    file_stream = BytesIO(file_data)
                    rawPR = pd.concat([rawPR, pd.read_csv(file_stream)])
        totalMailParsed += 1

338/1330, GRN: 398, PR: 398                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

abort: command: FETCH => socket error: EOF

In [ ]:
mail.logout()

('BYE', [b'LOGOUT Requested'])

In [ ]:
grn = rawGRN.dropna(how="all", axis=1)
# grn.to_csv("GRN (Swiggy).csv")

In [ ]:
pr = rawPR.dropna(how="all", axis=1)
# pr.to_csv("PR.csv")

In [ ]:
df = grn.merge(pr, how="outer", left_on="Invoice Number", right_on="Invoice Number")

In [ ]:
def check(df, name):
    return (df[f"{name}_x"] == df[f"{name}_y"]).sum() / len(df) * 100


for i in df.columns[df.columns.str.endswith("_y")]:
    column_name = i[:-2]
    result = check(df, column_name)
    if result == 100:
        df.drop(columns=[i], inplace=True)

In [ ]:
df.drop(labels=["Document Type_x", "Document Type_y"], axis=1, inplace=True)

In [ ]:
df.rename(columns={"Quantity": "PR Quantity", "Amount": "PR Amount"}, inplace=True)
df.columns = df.columns.str.replace("_x", "")

In [ ]:
df.to_csv("GRN PR (Swiggy).csv", index=False)